# ***PDF Question Answering with RAG***
## Retrieval-Augmented Generation Project
*Author:* Layan Mousa

### Description
This notebook builds a Retrieval-Augmented Generation system that answers questions from a PDF document. The workflow extracts text, divides the document into chunks, creates embeddings, stores them in ChromaDB, retrieves relevant context, and generates grounded answers using FLAN-T5.

### Source Document
The project uses the ISO/IEC 27001:2022 standard as the source document for retrieval and question answering.

### Project Goal
The goal is to generate answers based only on information retrieved from the provided document and reduce hallucination when the requested information is unavailable.

---

# **1. Import and Install Libraries**

### Purpose
Load the Python libraries required for PDF extraction, text processing, embeddings, vector storage, retrieval, and language-model generation.

In [1]:
import sys

packages = [
    "pypdf",
    "sentence-transformers",
    "chromadb",
    "transformers==4.55.4",
    "sentencepiece",
    "accelerate"
]

!{sys.executable} -m pip install -q {" ".join(packages)}

print(" All required libraries are installed successfully.")

 All required libraries are installed successfully.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

import chromadb

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

import torch

c:\Users\layan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **2. Load the PDF Document**

### Purpose
Load the selected source document and create a PDF reader object.

In [3]:
pdf_path = "iso27001.pdf"

reader = PdfReader(pdf_path)

## 2.1 Check the Number of Pages

Display the total number of pages in the PDF document.

In [ ]:
number_of_pages = len(reader.pages)

print("Number of pages:", number_of_pages)

Number of pages: 26


# **3. Extract Text from the PDF**

### Purpose
Extract readable text from every page and combine it into one document string.

In [5]:
document_text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        document_text += page_text + "\n"

## 3.1 Display an Extracted Text Sample

Display a sample of the extracted text to verify that the PDF was read correctly.

In [6]:
sample_size = 1000

print(document_text[:sample_size])

Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001
Third edition  
2022-10
Reference number 
ISO/IEC 27001:2022(E)
© ISO/IEC 2022
--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---
ii
ISO/IEC 27001:2022(E)
COPYRIGHT PROTECTED DOCUMENT
©  ISO/IEC 2022
All rights reserved. Unless otherwise specified, or required in the context of its implementation, no part of this publication may 
be reproduced or utilized otherwise in any form or by any means, electronic or mechanical, including photocopying, or posting on 
the internet or an intranet, without prior written permission. Permission can be requested from either ISO at the address below 
or ISO’s member body in the country of the requester.
ISO copyright office
CP 401 • Ch. de Blandonnet 8
CH-12

## 3.2 Check Extraction Quality

Measure the extracted text length and inspect whether useful content was successfully obtained.

In [7]:
text_length = len(document_text)
word_count = len(document_text.split())

print("Number of characters:", text_length)
print("Number of words:", word_count)

Number of characters: 57197
Number of words: 6748


## 3.3 Final Extraction Output

Display the final document information and a short text sample.

In [8]:
sample_size = 300

print("Document loaded successfully.")
print("Number of pages:", len(reader.pages))
print("Extracted characters:", len(document_text))
print("Sample:", document_text[:sample_size])

Document loaded successfully.
Number of pages: 26
Extracted characters: 57197
Sample: Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001


# **4. Document Chunking**

### Purpose
Split the extracted document into smaller overlapping chunks so that relevant passages can be retrieved efficiently.

- Chunk size: 500 characters
- Chunk overlap: 16 characters

In [9]:
chunk_size = 500
chunk_overlap = 16

chunks = []

start = 0

while start < len(document_text):
    end = start + chunk_size

    chunk = document_text[start:end]

    chunks.append(chunk)

    start += chunk_size - chunk_overlap

## 4.1 Review Generated Chunks

Display the number of chunks, the first chunk length, and a sample of the generated text.

In [10]:
print("Number of chunks:", len(chunks))
print("First chunk length:", len(chunks[0]))
print("\nFirst chunk sample:")
print(chunks[0])

Number of chunks: 119
First chunk length: 500

First chunk sample:
Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001
Third edition  
2022-10
Reference number 
ISO/IEC 27001:2022(E)
© ISO/IEC 2022
--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---
ii
ISO/IEC 27001:2022(E)
COPYRIGHT PROTECTED DOCUMENT
©  ISO/IEC 2022



## 4.2 Verify Chunk Overlap

Confirm that consecutive chunks share the expected overlapping text.

In [11]:
print("End of Chunk 1:")
print(chunks[0][-50:])

print("\nStart of Chunk 2:")
print(chunks[1][:50])

End of Chunk 1:
2(E)
COPYRIGHT PROTECTED DOCUMENT
©  ISO/IEC 2022


Start of Chunk 2:
©  ISO/IEC 2022
All rights reserved. Unless otherw


# **5. Create Embeddings**

### Purpose
Convert each text chunk into a numerical vector that represents its semantic meaning.

## 5.1 Load the Embedding Model

Load the Sentence Transformer model used to generate semantic embeddings.

In [12]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

## 5.2 Generate Chunk Embeddings

Create one embedding vector for every document chunk.

In [13]:
chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

## 5.3 Verify Embedding Output

Confirm that the number and dimensions of the generated embeddings match the chunks.

In [14]:
print(f"Number of chunks: {len(chunks)}")
print(f"Number of embeddings: {len(chunk_embeddings)}")
print(f"Embedding shape: {chunk_embeddings.shape}")

print("\nFirst embedding (first 10 values):")
print(chunk_embeddings[0][:10])

Number of chunks: 119
Number of embeddings: 119
Embedding shape: (119, 384)

First embedding (first 10 values):
[-0.02654819  0.01417822 -0.0905083  -0.09985209  0.04653104  0.01956914
  0.08642682  0.03603141  0.05179968  0.00656525]


# **6. Store Data in ChromaDB**

### Purpose
Store the document chunks and their embeddings in a persistent vector database for semantic retrieval.

## 6.1 Create a Persistent ChromaDB Client

Initialize a persistent ChromaDB client so the vector database can be saved locally.

In [15]:


chroma_client = chromadb.PersistentClient(
    path="chroma_db"
)

## 6.2 Create the Collection

Create a collection that organizes the document chunks and embeddings.

In [16]:
try:
    chroma_client.delete_collection(name="rag_documents")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="rag_documents"
)

## 6.3 Prepare Data for Storage

Create unique chunk identifiers and metadata before inserting the data into ChromaDB.

In [17]:
chunk_ids = [f"chunk_{i}" for i in range(len(chunks))]

chunk_metadata = [
    {
        "chunk_index": i,
        "source": "RAG_Sample_Knowledge_Base.pdf"
    }
    for i in range(len(chunks))
]

## 6.4 Store Chunks and Embeddings

Add the chunks, embeddings, identifiers, and metadata to the vector database.

In [18]:
collection.add(
    ids=chunk_ids,
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    metadatas=chunk_metadata
)

## 6.5 Verify Stored Data

Confirm that the number of stored records matches the number of generated chunks.

In [19]:
stored_chunks = collection.count()

print("Stored chunks:", stored_chunks)

Stored chunks: 119


# **7. Semantic Retrieval**

### Purpose
Convert the user question into an embedding and retrieve the most semantically relevant document chunks.

In [20]:
question = "What is the main objective of the document?"

question_embedding = embedding_model.encode(
    question,
    convert_to_numpy=True
)

## 7.1 Retrieve the Top Relevant Chunks

Query ChromaDB and return the three chunks with the smallest semantic distances.

In [21]:
top_k = 3

results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=top_k,
    include=["documents", "distances", "metadatas"]
)

## 7.2 Display Retrieved Context

Display the retrieved passages and their distance scores for inspection.

In [22]:
print("Question:")
print(question)

print("\nTop Retrieved Chunks:\n")

for i in range(top_k):
    print(f"Result {i+1}")
    print(f"Distance Score: {results['distances'][0][i]:.4f}")
    print(results["documents"][0][i])
    print("-" * 80)

Question:
What is the main objective of the document?

Top Retrieved Chunks:

Result 1
Distance Score: 1.0900
opedia: available at h t t p s :// www .electropedia  .org/ 
4  Context of the organization
4.1  Understanding the organization and its context
The organization shall determine external and internal issues that are relevant to its purpose and that 
affect its ability to achieve the intended outcome(s) of its information security management system.
NOTE Determining these issues refers to establishing the external and internal context of the organization 
considered in Clause 5.4.1 of ISO 31000:20
--------------------------------------------------------------------------------
Result 2
Distance Score: 1.1278
 functions and levels.
The information security objectives shall:
a) be consistent with the information security policy;
b) be measurable (if practicable);
c) take into account applicable information security requirements, and results from risk assessment 
and risk treatment;

# **8. Answer Generation**

### Purpose
Combine the retrieved chunks into context and use a language model to generate a grounded answer.

In [23]:
retrieved_chunks = results["documents"][0]

retrieved_context = "\n\n".join(retrieved_chunks)

print("Retrieved Context:\n")
print(retrieved_context)

Retrieved Context:

opedia: available at h t t p s :// www .electropedia  .org/ 
4  Context of the organization
4.1  Understanding the organization and its context
The organization shall determine external and internal issues that are relevant to its purpose and that 
affect its ability to achieve the intended outcome(s) of its information security management system.
NOTE Determining these issues refers to establishing the external and internal context of the organization 
considered in Clause 5.4.1 of ISO 31000:20

 functions and levels.
The information security objectives shall:
a) be consistent with the information security policy;
b) be measurable (if practicable);
c) take into account applicable information security requirements, and results from risk assessment 
and risk treatment;
d) be monitored;
e) be communicated; 
f) be updated as appropriate;
g) be available as documented information.
The organization shall retain documented information on the information security objective

In [24]:
retrieved_context

'opedia: available at h t t p s :// www .electropedia  .org/ \n4  Context of the organization\n4.1  Understanding the organization and its context\nThe organization shall determine external and internal issues that are relevant to its purpose and that \naffect its ability to achieve the intended outcome(s) of its information security management system.\nNOTE Determining these issues refers to establishing the external and internal context of the organization \nconsidered in Clause 5.4.1 of ISO 31000:20\n\n functions and levels.\nThe information security objectives shall:\na) be consistent with the information security policy;\nb) be measurable (if practicable);\nc) take into account applicable information security requirements, and results from risk assessment \nand risk treatment;\nd) be monitored;\ne) be communicated; \nf) be updated as appropriate;\ng) be available as documented information.\nThe organization shall retain documented information on the information security objectives

## 8.1 Create the Prompt

Construct a prompt that instructs the model to answer only from the retrieved document context.

In [25]:
prompt = f"""
You are an AI assistant.

Use ONLY the context below to answer the user's question.

Rules:
1. Answer only from the provided context.
2. Do not invent or infer information.
3. If the answer is not available in the context, reply exactly:
"The information is not available in the provided document."

Context:
{retrieved_context}

Question:
{question}

Answer:
"""

## 8.2 Prepare the Language Model

Load the tokenizer and model components required for FLAN-T5 answer generation.

## 8.3 Load the Tokenizer and Model Dependencies

Import and verify the Transformers components used by the language model.

In [26]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

In [27]:
import transformers
import tokenizers
import sentencepiece

print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("sentencepiece:", sentencepiece.__version__)

transformers: 4.55.4
tokenizers: 0.21.4
sentencepiece: 0.2.2


In [28]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


## 8.4 Load the FLAN-T5 Model

Load the pretrained FLAN-T5 model used to generate the final answer.

In [29]:
from transformers import AutoModelForSeq2SeqLM

llm_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

print("Model loaded successfully!")

Model loaded successfully!


## 8.5 Generate the Final Answer

Tokenize the prompt, generate the response, and decode the model output.

In [30]:
question = "What is the main objective of the document?"

results = collection.query(
    query_texts=[question],
    n_results=3
)

retrieved_chunks = results["documents"][0]

context = "\n\n".join(retrieved_chunks)

prompt = f"""
Answer the question using only the provided context.

If the answer is not available in the context, say:
"The information is not available in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

In [31]:
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = llm_model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=False
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
What is the main objective of the document?

Answer:
Understanding the organization and its context


# **9. Test the RAG System**

### Purpose
Evaluate the complete RAG pipeline using ten questions:

- Five questions with answers available in the document.
- Three detailed questions that require contextual understanding.
- Two questions whose answers are unavailable and should not trigger hallucination.

In [32]:
test_questions = [

    # ==========================
    # Available Questions (5)
    # ==========================

    {
        "category": "Available",
        "question": "What does ISO/IEC 27001 specify?"
    },

    {
        "category": "Available",
        "question": "What is the title of Clause 4?"
    },

    {
        "category": "Available",
        "question": "What is the title of Clause 5?"
    },

    {
        "category": "Available",
        "question": "What is the title of Clause 10?"
    },

    {
        "category": "Available",
        "question": "What document contains the terms and definitions?"
    },

    # ==========================
    # Detailed Questions (3)
    # ==========================

    {
        "category": "Detailed",
        "question": "Summarize the purpose of the information security management system."
    },

    {
        "category": "Detailed",
        "question": "Explain the purpose of internal audits."
    },

    {
        "category": "Detailed",
        "question": "What should an organization do when a nonconformity occurs?"
    },

    # ==========================
    # Unavailable Questions (2)
    # ==========================

    {
        "category": "Unavailable",
        "question": "Who created ISO/IEC 27001?"
    },

    {
        "category": "Unavailable",
        "question": "What is the certification cost of ISO/IEC 27001?"
    }

]

## 9.1 Run the Test Questions

Execute every test question, retrieve context, generate an answer, and display the semantic distance scores.

In [33]:
test_results = []

for index, item in enumerate(test_questions, start=1):
    question = item["question"]

    # Convert the question into an embedding
    question_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True
    )

    # Retrieve the top 3 relevant chunks
    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=3,
        include=["documents", "distances", "metadatas"]
    )

    retrieved_chunks = results["documents"][0]
    distances = results["distances"][0]

    context = "\n\n".join(retrieved_chunks)

    # Create a grounded prompt
    prompt = f"""
Answer the question using only the provided context.

Rules:
1. Use only information explicitly available in the context.
2. Do not use outside knowledge.
3. Do not invent or infer missing information.
4. If the answer is not clearly available, reply exactly:
"The information is not available in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

    # Convert the prompt into model inputs
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Generate the grounded answer
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Save the test result
    test_results.append(
        {
            "test_number": index,
            "category": item["category"],
            "question": question,
            "answer": answer,
            "retrieved_chunks": retrieved_chunks,
            "distances": distances
        }
    )

    # Display the result
    print("=" * 100)
    print(f"Test {index}")
    print(f"Category: {item['category']}")
    print(f"Question: {question}")

    print("\nAnswer:")
    print(answer)

    print("\nRetrieved Chunk Distances:")
    for rank, distance in enumerate(distances, start=1):
        print(f"{rank}. {distance:.4f}")

    print()

Test 1
Category: Available
Question: What does ISO/IEC 27001 specify?

Answer:
Information technology — Security techniques — Information security management systems

Retrieved Chunk Distances:
1. 0.4842
2. 0.5500
3. 0.5795

Test 2
Category: Available
Question: What is the title of Clause 4?

Answer:
Information security management system

Retrieved Chunk Distances:
1. 1.3972
2. 1.4392
3. 1.4618

Test 3
Category: Available
Question: What is the title of Clause 5?

Answer:
ISO/IEC 27002:2022[1]

Retrieved Chunk Distances:
1. 1.4339
2. 1.4594
3. 1.4680

Test 4
Category: Available
Question: What is the title of Clause 10?

Answer:
Annex A (normative) Information security controls

Retrieved Chunk Distances:
1. 1.3792
2. 1.3939
3. 1.4136

Test 5
Category: Available
Question: What document contains the terms and definitions?

Answer:
ISO/IEC 27000

Retrieved Chunk Distances:
1. 0.9464
2. 1.0695
3. 1.1045

Test 6
Category: Detailed
Question: Summarize the purpose of the information security 

In [34]:
document_text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        document_text += page_text + "\n"

In [35]:
print("Document loaded successfully.")
print("Number of pages:", len(reader.pages))
print("Extracted characters:", len(document_text))
print("Sample:", document_text[:300])
print("Sample:", document_text[:300])

Document loaded successfully.
Number of pages: 26
Extracted characters: 57197
Sample: Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001
Sample: Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001


# **10. Conclusion**

This project implemented a complete Retrieval-Augmented Generation pipeline for PDF question answering. The system extracted text from ISO/IEC 27001:2022, divided it into overlapping chunks, generated semantic embeddings, stored them in ChromaDB, retrieved relevant passages, and generated answers with FLAN-T5.

The test set demonstrated the strengths and limitations of fixed-size chunking. The system answered several available and detailed questions correctly and rejected some unavailable questions, while questions involving clause titles showed that structure-aware or adaptive chunking would be a valuable future improvement.